# Hafta 4 · Tek Kübit Kapıları I: X, Y, Z ve H
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

Bu hafta her kapıyı **üç gözle** inceliyoruz: **matris** (hesap), **Bloch küresinde dönüş** (görüntü) ve **Qiskit kodu** (program).
Her kapı için aynı şablonu izleyeceğiz: matris × vektör → altı temel durum tablosu → devre çizimi → Bloch öncesi/sonrası + dönüş yolu → film şeridi → Qiskit Statevector + histogram → NumPy doğrulaması.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar (`plot_gate_on_bloch`, `plot_film`, `gate_table`) | 4 dk |
| A | Kapı = dönüş: matristen eksen ve açıyı okumak | 5 dk |
| B | X kapısı (x ekseni, 180°) | 6 dk |
| C | Y kapısı (y ekseni, 180°) | 5 dk |
| D | Z kapısı (z ekseni, 180°) | 5 dk |
| E | H kapısı (çapraz eksen, 180°) | 7 dk |
| F | Kapı özdeşlikleri ve `Operator(...).equiv` | 6 dk |
| G | Ardışık kapılar: H → Z → H yolculuğu | 5 dk |
| H | Tersinirlik: kapıyı geri almak | 3 dk |
| I | Alıştırmalar (8 adet, `assert` ile) | ödev |

Etkileşimli sürüm: ders sayfasındaki **`Hafta04_Bloch_Simulatoru.html`** dosyasını tarayıcıda açın; aynı dönüşleri fareyle döndürerek izleyebilirsiniz.

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, Operator

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY, LIGHT = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6", "#E8EEF6"
CSTYLE = {"backgroundcolor": "#FFFFFF", "gatefacecolor": LIGHT, "gatetextcolor": "#111111", "linecolor": NAVY, "textcolor": "#111111",
          "displaycolor": {"h": [BLUE, "#FFFFFF"], "x": [NAVY, "#FFFFFF"], "y": [NAVY, "#FFFFFF"], "z": [NAVY, "#FFFFFF"],
                           "cx": [NAVY, "#FFFFFF"], "measure": [ORANGE, "#FFFFFF"]}}
aer = AerSimulator(seed_simulator=2026)

# ---- Kapılar (2. haftadaki matrisler) ----
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
GATES = {"I": I, "X": X, "Y": Y, "Z": Z, "H": H}

# ---- Altı temel durum ----
r = 1 / np.sqrt(2)
STATES = {"|0⟩": np.array([1, 0], complex), "|1⟩": np.array([0, 1], complex),
          "|+⟩": np.array([r, r], complex), "|−⟩": np.array([r, -r], complex),
          "|+i⟩": np.array([r, 1j*r], complex), "|−i⟩": np.array([r, -1j*r], complex)}

def state_to_bloch(amps):
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def plot_bloch(amps_list, titles=None):
    """Kübit durumlarını yan yana Bloch küresinde çizer (1-2. haftadaki fonksiyon)."""
    if np.ndim(amps_list) == 1: amps_list = [amps_list]
    k = len(amps_list); fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, amps in enumerate(amps_list):
        ax = fig.add_subplot(1, k, i+1, projection="3d")
        _sphere(ax, titles[i] if titles else None)
        _arrow(ax, state_to_bloch(amps), BLUE)
    plt.show()

def _sphere(ax, title=None):
    """Boş Bloch küresi (ders boyunca aynı stil)."""
    ax.set_box_aspect((1,1,1), zoom=1.3); ax.computed_zorder = False
    u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
    ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)),
                    color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
    t = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8); ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
    for d in [(1,0,0), (0,1,0), (0,0,1)]: ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
    for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"), ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
        ax.text(*p, s, ha="center", va="center", fontsize=9, color=NAVY)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1); ax.view_init(elev=18, azim=30); ax.set_axis_off()
    if title: ax.set_title(title, fontsize=10.5, color=NAVY)

def _arrow(ax, v, color):
    ax.plot([0, v[0]], [0, v[1]], [0, v[2]], color=color, lw=3); ax.scatter(*[[c] for c in v], color=color, s=55, depthshade=False)
print("hazır")

### Yardımcı fonksiyonlar: kapıyı küre üzerinde göstermek
Aşağıdaki fonksiyonlar bu haftanın ana araçlarıdır:
- `bloch_rotation(U)` → kapının Bloch küresindeki **dönüş eksenini ve açısını** matristen hesaplar
- `rotate(v, n, a)` → bir Bloch vektörünü n ekseni etrafında a radyan döndürür (Rodrigues formülü)
- `plot_gate_on_bloch(U, start_states)` → her başlangıç durumu için **önce (gri) / sonra (mavi) / dönüş yolu (turuncu)**
- `plot_film(U, start)` → dönüşün 0°, 45°, 90°, 135°, 180° karelerini gösteren **film şeridi**
- `gate_table(U)` → altı temel durum üzerindeki etki tablosu

In [ ]:
def rotate(v, n, a):
    """Rodrigues: v'yi birim n ekseni etrafında a radyan döndürür (sağ el kuralı)."""
    v, n = np.asarray(v, float), np.asarray(n, float) / np.linalg.norm(n)
    return v*np.cos(a) + np.cross(n, v)*np.sin(a) + n*np.dot(n, v)*(1 - np.cos(a))

def bloch_rotation(U):
    """Tek kübit üniter U için (eksen n, açı θ) döndürür.
    Fikir: global fazı atınca U = cos(θ/2)·I − i·sin(θ/2)·(nx·X + ny·Y + nz·Z)."""
    U = np.asarray(U, complex)
    V = U / np.sqrt(np.linalg.det(U))                      # global fazı at (det = 1)
    c = np.real(np.trace(V)) / 2                           # cos(θ/2)
    s = np.array([np.real(1j*np.trace(V @ P)) / 2 for P in (X, Y, Z)])   # sin(θ/2)·n
    if np.linalg.norm(s) < 1e-12: return np.array([0, 0, 1.0]), 0.0      # birim kapı: dönüş yok
    return s / np.linalg.norm(s), 2*np.arctan2(np.linalg.norm(s), c)

def _axis(ax, n):
    ax.plot(*[[-1.15*n[i], 1.15*n[i]] for i in range(3)], color=NAVY, lw=1.8, ls="-.")

def plot_gate_on_bloch(U, start_states, name="U"):
    """Her başlangıç durumu için kapının etkisini gösterir: önce, sonra, dönüş yolu ve dönüş ekseni."""
    if isinstance(start_states, dict): items = list(start_states.items())
    else: items = [(k, STATES[k]) if isinstance(k, str) else (f"ψ{i}", k) for i, k in enumerate(start_states)]
    n, th = bloch_rotation(U)
    fig = plt.figure(figsize=(3.7*len(items), 3.9))
    for i, (lab, psi) in enumerate(items):
        ax = fig.add_subplot(1, len(items), i+1, projection="3d")
        _sphere(ax, f"{lab} → {name}{lab}"); _axis(ax, n)
        v0, v1 = state_to_bloch(psi), state_to_bloch(U @ psi)
        path = np.array([rotate(v0, n, a) for a in np.linspace(0, th, 60)])
        ax.plot(path[:, 0], path[:, 1], path[:, 2], color=ORANGE, lw=2.2, ls=":")
        _arrow(ax, v0, GRAY); _arrow(ax, v1, BLUE)
    plt.show()

def plot_film(U, start="|0⟩", degs=(0, 45, 90, 135, 180)):
    """Dönüşün ara karelerini (film şeridi) çizer. Son kare U @ ψ ile aynı noktadır."""
    psi = STATES[start] if isinstance(start, str) else start
    n, th = bloch_rotation(U); v0 = state_to_bloch(psi)
    fig = plt.figure(figsize=(3.3*len(degs), 3.5))
    for i, d in enumerate(degs):
        a = np.radians(d) * np.sign(th) if th else 0
        ax = fig.add_subplot(1, len(degs), i+1, projection="3d"); _sphere(ax, f"{d}°"); _axis(ax, n)
        path = np.array([rotate(v0, n, t) for t in np.linspace(0, a, 30)])
        ax.plot(path[:, 0], path[:, 1], path[:, 2], color=ORANGE, lw=2.2, ls=":")
        _arrow(ax, v0, GRAY); _arrow(ax, rotate(v0, n, a), BLUE)
    plt.show()

def same_state(a, b):
    """Global faz farkını yok sayarak iki durumu karşılaştırır (2. hafta, Alıştırma 2)."""
    return bool(np.isclose(abs(np.vdot(a, b)), 1.0))

def name_of(psi):
    """Durumu altı temel durumdan biriyle eşleştirir; global fazı da yazar."""
    for k, s in STATES.items():
        if same_state(psi, s):
            ph = np.vdot(s, psi)                     # psi = ph · s
            ph = complex(np.round(ph, 6))
            pre = {1: "", -1: "−", 1j: "i", -1j: "−i"}.get(ph, f"e^(i·{np.degrees(np.angle(ph)):.0f}°)")
            return pre + k
    return "?"

def gate_table(U, name="U"):
    print(f"{'girdi':>6} | {'çıktı vektörü':>22} | çıktı durumu | Bloch (x, y, z)")
    print("-"*72)
    for k, s in STATES.items():
        out = U @ s
        print(f"{k:>6} | {str(np.round(out, 3)):>22} | {name_of(out):>12} | {np.round(state_to_bloch(out), 3)}")

def counts_bar(counts, title="Histogram"):
    plt.figure(figsize=(4, 2.6)); ks = ["0", "1"]
    plt.bar(ks, [counts.get(k, 0) for k in ks], color=NAVY, width=0.5)
    for i, k in enumerate(ks): plt.text(i, counts.get(k, 0) + 10, counts.get(k, 0), ha="center")
    plt.title(title, color=NAVY, fontsize=11); plt.ylabel("sayım"); plt.show()

def run_counts(qc, shots=1000):
    qm = qc.copy(); qm.measure_all()
    return aer.run(transpile(qm, aer), shots=shots).result().get_counts()

# Hızlı kontrol: matrisle uygulanan kapı = Bloch vektörünü eksen etrafında döndürmek
for g in "XYZH":
    n, th = bloch_rotation(GATES[g])
    ok = all(np.allclose(state_to_bloch(GATES[g] @ s), rotate(state_to_bloch(s), n, th)) for s in STATES.values())
    print(f"{g}: eksen = {np.round(n, 3)}, açı = {np.degrees(th):.0f}°, matris ↔ dönüş tutarlı: {ok}")

---
## A · Kapı = dönüş
2\. haftada kapının **üniter bir matris** olduğunu gördük: `q_yeni = U @ q`. Üniter matris vektörün uzunluğunu korur; bu yüzden Bloch noktası **kürenin yüzeyinde kalır**. Yüzeyde kalan ve noktalar arası açıları koruyan her hareket bir **dönüştür**. Sonuç:

> **Her tek kübit kapısı, Bloch küresinde bir eksen etrafında bir dönüştür.**

| Kapı | Matris | Dönüş ekseni | Açı | Yazılım benzetmesi |
|---|---|---|---|---|
| X | [[0, 1], [1, 0]] | x | 180° | NOT (bit çevirme) |
| Y | [[0, −i], [i, 0]] | y | 180° | NOT + işaret/faz |
| Z | [[1, 0], [0, −1]] | z | 180° | işaret çevirme (`-x`) |
| H | (1/√2)[[1, 1], [1, −1]] | (x+z)/√2 | 180° | baz değiştirici (kodlama dönüştürücü) |

**Eksen üzerindeki durumlar yerinden kıpırdamaz** (yalnızca global faz alabilir). Eksene dik olanlar tam karşı tarafa geçer.

In [ ]:
# Bir kapıyı |0⟩'a uygulamak: hesap + görüntü
psi = STATES["|0⟩"]
print("X|0⟩ =", X @ psi, " Bloch:", state_to_bloch(X @ psi))
plot_gate_on_bloch(X, ["|0⟩"], "X")

---
## B · X kapısı: x ekseni etrafında 180°

**Matris:** X = [[0, 1], [1, 0]] → iki genliğin **yerini değiştirir**: X·[α, β] = [β, α].

**Yazılım benzetmesi:** Klasik **NOT** (`b ^= 1`). |0⟩ ↔ |1⟩. Ama X klasik NOT'tan fazlasını yapar: süperpozisyondaki genlikleri de takas eder.

**Bloch:** x ekseni etrafında 180°. Eksen üzerindeki |+⟩ ve |−⟩ yerinde kalır (|−⟩ yalnızca −1 global faz alır). z ve y koordinatlarının işareti değişir: (x, y, z) → (x, −y, −z).

### X.1 · Matris × vektör (elle hesap → NumPy)

In [ ]:
alpha, beta = 0.6, 0.8                       # genel bir gerçek durum
q = np.array([alpha, beta], dtype=complex)
print("X =\n", X)
print("X · [0.6, 0.8] =", X @ q)
print("olasılıklar önce:", np.round(abs(q)**2, 3), " sonra:", np.round(abs(X @ q)**2, 3))
print("üniter mi?", np.allclose(X.conj().T @ X, I))

### X.2 · Altı temel durum üzerindeki etki tablosu

In [ ]:
gate_table(X, 'X')

### X.3 · Devre sembolü (Qiskit)

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
qc.draw("mpl", style=CSTYLE, scale=1.2)

### X.4 · Bloch küresi: önce (gri), sonra (mavi), dönüş yolu (turuncu), dönüş ekseni (lacivert)

In [ ]:
plot_gate_on_bloch(X, ["|0⟩", "|+i⟩", "|+⟩"], "X")
# genel (temel olmayan) bir durum:
psi = np.array([np.cos(np.pi/6), np.exp(1j*np.pi/4)*np.sin(np.pi/6)])
plot_gate_on_bloch(X, {"ψ": psi}, "X")

### X.5 · Film şeridi: dönüşün ara kareleri

In [ ]:
plot_film(X, "|0⟩")

### X.6 · Qiskit: Statevector ve histogram
X|0⟩ = |1⟩ → 1000 shot'ın hepsi 1.

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
sv = Statevector(qc)
print("Statevector:", np.round(sv.data, 3))
print("Olasılıklar:", sv.probabilities_dict())
counts_bar(run_counts(qc), "X devresi, 1000 shot")

### X.7 · NumPy doğrulaması: Qiskit = matris = dönüş

In [ ]:
numpy_out = X @ STATES["|0⟩"]
print("NumPy :", np.round(numpy_out, 3))
print("Qiskit ile aynı mı?", np.allclose(numpy_out, sv.data))
only = QuantumCircuit(1); only.x(0)            # yalnızca kapının kendisi
print("Qiskit operatörü = bizim matrisimiz mi?", np.allclose(Operator(only).data, X))
n, th = bloch_rotation(X)
print("Matristen okunan eksen:", np.round(n, 3), " açı:", round(np.degrees(th)), "°")

---
## C · Y kapısı: y ekseni etrafında 180°

**Matris:** Y = [[0, −i], [i, 0]] → Y·[α, β] = [−iβ, iα]. Genlikleri takas eder **ve** i / −i faz çarpanları ekler.

**Yazılım benzetmesi:** "NOT + faz etiketi". Ölçüm istatistiği X ile aynıdır (|0⟩ → hep 1), ama fazlar farklıdır. Y = i·X·Z olduğu için "önce işaret çevir, sonra NOT uygula (ve global faz)" diye düşünebilirsiniz.

**Bloch:** y ekseni etrafında 180°: (x, y, z) → (−x, y, −z). |+i⟩ ve |−i⟩ yerinde kalır.

### Y.1 · Matris × vektör (elle hesap → NumPy)

In [ ]:
alpha, beta = 0.6, 0.8                       # genel bir gerçek durum
q = np.array([alpha, beta], dtype=complex)
print("Y =\n", Y)
print("Y · [0.6, 0.8] =", Y @ q)
print("olasılıklar önce:", np.round(abs(q)**2, 3), " sonra:", np.round(abs(Y @ q)**2, 3))
print("üniter mi?", np.allclose(Y.conj().T @ Y, I))

### Y.2 · Altı temel durum üzerindeki etki tablosu

In [ ]:
gate_table(Y, 'Y')

### Y.3 · Devre sembolü (Qiskit)

In [ ]:
qc = QuantumCircuit(1)
qc.y(0)
qc.draw("mpl", style=CSTYLE, scale=1.2)

### Y.4 · Bloch küresi: önce (gri), sonra (mavi), dönüş yolu (turuncu), dönüş ekseni (lacivert)

In [ ]:
plot_gate_on_bloch(Y, ["|0⟩", "|+⟩", "|+i⟩"], "Y")
# genel (temel olmayan) bir durum:
psi = np.array([np.cos(np.pi/6), np.exp(1j*np.pi/4)*np.sin(np.pi/6)])
plot_gate_on_bloch(Y, {"ψ": psi}, "Y")

### Y.5 · Film şeridi: dönüşün ara kareleri

In [ ]:
plot_film(Y, "|0⟩")

### Y.6 · Qiskit: Statevector ve histogram
Y|0⟩ = i|1⟩ → genlik sanal ama olasılık |i|² = 1.

In [ ]:
qc = QuantumCircuit(1)
qc.y(0)
sv = Statevector(qc)
print("Statevector:", np.round(sv.data, 3))
print("Olasılıklar:", sv.probabilities_dict())
counts_bar(run_counts(qc), "Y devresi, 1000 shot")

### Y.7 · NumPy doğrulaması: Qiskit = matris = dönüş

In [ ]:
numpy_out = Y @ STATES["|0⟩"]
print("NumPy :", np.round(numpy_out, 3))
print("Qiskit ile aynı mı?", np.allclose(numpy_out, sv.data))
only = QuantumCircuit(1); only.y(0)            # yalnızca kapının kendisi
print("Qiskit operatörü = bizim matrisimiz mi?", np.allclose(Operator(only).data, Y))
n, th = bloch_rotation(Y)
print("Matristen okunan eksen:", np.round(n, 3), " açı:", round(np.degrees(th)), "°")

---
## D · Z kapısı: z ekseni etrafında 180°

**Matris:** Z = [[1, 0], [0, −1]] → Z·[α, β] = [α, −β]. Yalnızca |1⟩ genliğinin **işaretini çevirir**.

**Yazılım benzetmesi:** **İşaret çevirme** (`x = -x`, ama sadece "1" kolunda). Olasılıklar hiç değişmez: |−β|² = |β|². Bu yüzden Z'nin etkisi doğrudan ölçümle **görülmez**; ancak önce/sonra H ile sarılırsa görünür.

**Bloch:** z ekseni etrafında 180°: (x, y, z) → (−x, −y, z). Kutuplar (|0⟩, |1⟩) yerinde kalır; ekvator üzerindeki noktalar karşı tarafa geçer: |+⟩ ↔ |−⟩, |+i⟩ ↔ |−i⟩.

### Z.1 · Matris × vektör (elle hesap → NumPy)

In [ ]:
alpha, beta = 0.6, 0.8                       # genel bir gerçek durum
q = np.array([alpha, beta], dtype=complex)
print("Z =\n", Z)
print("Z · [0.6, 0.8] =", Z @ q)
print("olasılıklar önce:", np.round(abs(q)**2, 3), " sonra:", np.round(abs(Z @ q)**2, 3))
print("üniter mi?", np.allclose(Z.conj().T @ Z, I))

### Z.2 · Altı temel durum üzerindeki etki tablosu

In [ ]:
gate_table(Z, 'Z')

### Z.3 · Devre sembolü (Qiskit)

In [ ]:
qc = QuantumCircuit(1)
qc.z(0)
qc.draw("mpl", style=CSTYLE, scale=1.2)

### Z.4 · Bloch küresi: önce (gri), sonra (mavi), dönüş yolu (turuncu), dönüş ekseni (lacivert)

In [ ]:
plot_gate_on_bloch(Z, ["|+⟩", "|+i⟩", "|0⟩"], "Z")
# genel (temel olmayan) bir durum:
psi = np.array([np.cos(np.pi/6), np.exp(1j*np.pi/4)*np.sin(np.pi/6)])
plot_gate_on_bloch(Z, {"ψ": psi}, "Z")

### Z.5 · Film şeridi: dönüşün ara kareleri

In [ ]:
plot_film(Z, "|+⟩")

### Z.6 · Qiskit: Statevector ve histogram
Z|+⟩ = |−⟩ → ölçüm yine %50/%50; farkı görmek için ölçümden önce H.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)   # önce |+⟩ hazırla (Z'nin etkisi |0⟩ üzerinde görünmez)
qc.z(0)
sv = Statevector(qc)
print("Statevector:", np.round(sv.data, 3))
print("Olasılıklar:", sv.probabilities_dict())
counts_bar(run_counts(qc), "Z devresi, 1000 shot")

# Z'nin etkisini görünür yapmak: ölçümden önce H (|−⟩ → |1⟩)
qc2 = qc.copy(); qc2.h(0)
counts_bar(run_counts(qc2), "H·Z·H|0⟩ → hep 1")

### Z.7 · NumPy doğrulaması: Qiskit = matris = dönüş

In [ ]:
numpy_out = Z @ STATES["|+⟩"]
print("NumPy :", np.round(numpy_out, 3))
print("Qiskit ile aynı mı?", np.allclose(numpy_out, sv.data))
only = QuantumCircuit(1); only.z(0)            # yalnızca kapının kendisi
print("Qiskit operatörü = bizim matrisimiz mi?", np.allclose(Operator(only).data, Z))
n, th = bloch_rotation(Z)
print("Matristen okunan eksen:", np.round(n, 3), " açı:", round(np.degrees(th)), "°")

---
## E · H (Hadamard) kapısı: çapraz eksen etrafında 180°

**Matris:** H = (1/√2)[[1, 1], [1, −1]] → H·[α, β] = [(α+β)/√2, (α−β)/√2]. Toplam ve farkı alır.

**Yazılım benzetmesi:** **Baz değiştirici** (kodlama dönüştürücü): "z dilinde" yazılmış bilgiyi "x diline" çevirir ve tersi. |0⟩ → |+⟩, |1⟩ → |−⟩, |+⟩ → |0⟩, |−⟩ → |1⟩. Kendi tersidir (H·H = I), tıpkı iki kez uygulanan bir kodlama/çözme çifti gibi.

**Bloch:** x ile z'nin tam ortasındaki **(x+z)/√2 ekseni** etrafında 180°: (x, y, z) → (z, −y, x). x ile z yer değiştirir, y'nin işareti döner. Bu yüzden |+i⟩ → |−i⟩ (global faz ile).

### H.1 · Matris × vektör (elle hesap → NumPy)

In [ ]:
alpha, beta = 0.6, 0.8                       # genel bir gerçek durum
q = np.array([alpha, beta], dtype=complex)
print("H =\n", H)
print("H · [0.6, 0.8] =", H @ q)
print("olasılıklar önce:", np.round(abs(q)**2, 3), " sonra:", np.round(abs(H @ q)**2, 3))
print("üniter mi?", np.allclose(H.conj().T @ H, I))

### H.2 · Altı temel durum üzerindeki etki tablosu

In [ ]:
gate_table(H, 'H')

### H.3 · Devre sembolü (Qiskit)

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.draw("mpl", style=CSTYLE, scale=1.2)

### H.4 · Bloch küresi: önce (gri), sonra (mavi), dönüş yolu (turuncu), dönüş ekseni (lacivert)

In [ ]:
plot_gate_on_bloch(H, ["|0⟩", "|1⟩", "|+i⟩"], "H")
# genel (temel olmayan) bir durum:
psi = np.array([np.cos(np.pi/6), np.exp(1j*np.pi/4)*np.sin(np.pi/6)])
plot_gate_on_bloch(H, {"ψ": psi}, "H")

### H.5 · Film şeridi: dönüşün ara kareleri

In [ ]:
plot_film(H, "|0⟩")

### H.6 · Qiskit: Statevector ve histogram
H|0⟩ = |+⟩ → %50/%50.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
sv = Statevector(qc)
print("Statevector:", np.round(sv.data, 3))
print("Olasılıklar:", sv.probabilities_dict())
counts_bar(run_counts(qc), "H devresi, 1000 shot")

### H.7 · NumPy doğrulaması: Qiskit = matris = dönüş

In [ ]:
numpy_out = H @ STATES["|0⟩"]
print("NumPy :", np.round(numpy_out, 3))
print("Qiskit ile aynı mı?", np.allclose(numpy_out, sv.data))
only = QuantumCircuit(1); only.h(0)            # yalnızca kapının kendisi
print("Qiskit operatörü = bizim matrisimiz mi?", np.allclose(Operator(only).data, H))
n, th = bloch_rotation(H)
print("Matristen okunan eksen:", np.round(n, 3), " açı:", round(np.degrees(th)), "°")

---
## F · Kapı özdeşlikleri ve eşdeğer devreler
Derleyicilerin (transpiler) yaptığı en temel iş, bir devreyi **eşdeğer** ama daha kısa/uygun bir devreyle değiştirmektir. Bunun için özdeşlikleri bilmek gerekir.

| Özdeşlik | Anlamı | Bloch yorumu |
|---|---|---|
| X² = Y² = Z² = H² = I | her biri kendi tersidir | 180° + 180° = 360° = hiç dönmemek |
| HXH = Z, HZH = X | H, x ile z'nin rolünü değiştirir | "çevir → uygula → geri çevir" |
| HYH = −Y | −1 global faz | Bloch'ta HYH ile Y aynı dönüş |
| XZ = −ZX | sıra değişince işaret değişir | varış noktası aynı, global faz farklı |
| XY = iZ | iki 180° dönüş = bir 180° dönüş | global faz i |

`Operator(qc1).equiv(Operator(qc2))` **global faz farkını yok sayarak** karşılaştırır; `==` ise birebir eşitlik ister.

In [ ]:
def circ(ops):
    qc = QuantumCircuit(1)
    for o in ops: getattr(qc, o)(0) if o != "i" else qc.id(0)
    return qc

tests = [("HXH = Z", "hxh", "z"), ("HZH = X", "hzh", "x"), ("HYH = −Y", "hyh", "y"), ("X·X = I", "xx", "i"),
         ("H·H = I", "hh", "i"), ("XY = iZ (devrede önce Y, sonra X)", "yx", "z"), ("ZX = iY (önce X, sonra Z)", "xz", "y")]
print(f"{'özdeşlik':40s} equiv (faz hariç)   == (birebir)")
for name, a, b in tests:
    A, B = Operator(circ(a)), Operator(circ(b))
    print(f"{name:40s} {str(A.equiv(B)):18s} {A == B}")

⚠️ **Sıra kuralı (2. hafta):** Matematikte `X·Y` yazılan çarpım, devrede **önce Y sonra X** demektir. Yukarıdaki `"yx"` devresi X·Y matrisine karşılık gelir.

In [ ]:
# NumPy ile aynı özdeşlikler (global faz dahil)
print("H X H == Z   :", np.allclose(H @ X @ H, Z))
print("H Z H == X   :", np.allclose(H @ Z @ H, X))
print("H Y H == −Y  :", np.allclose(H @ Y @ H, -Y))
print("X Z == −Z X  :", np.allclose(X @ Z, -Z @ X))
print("X Y == i Z   :", np.allclose(X @ Y, 1j * Z))
print("Y   == i X Z :", np.allclose(Y, 1j * X @ Z))

# Eşdeğer devreleri çiz
fig, axs = plt.subplots(1, 2, figsize=(8, 1.6))
circ("hxh").draw("mpl", style=CSTYLE, ax=axs[0]); circ("z").draw("mpl", style=CSTYLE, ax=axs[1])
axs[0].set_title("H → X → H", color=NAVY); axs[1].set_title("= Z", color=NAVY); plt.show()

---
## G · Ardışık kapılar: Bloch üzerinde yolculuk
Bir devre, dönüşlerin **art arda** uygulanmasıdır. `plot_sequence` her adımda noktanın nereye gittiğini ve toplam yolu çizer.

In [ ]:
def plot_sequence(gates, start="|0⟩"):
    psi = STATES[start] if isinstance(start, str) else start
    k = len(gates) + 1
    fig = plt.figure(figsize=(3.4*k, 3.6))
    ax = fig.add_subplot(1, k, 1, projection="3d"); _sphere(ax, f"başlangıç {start}"); _arrow(ax, state_to_bloch(psi), BLUE)
    for i, g in enumerate(gates):
        U = GATES[g]; n, th = bloch_rotation(U); v0 = state_to_bloch(psi); psi = U @ psi
        ax = fig.add_subplot(1, k, i+2, projection="3d"); _sphere(ax, f"{i+1}) {g} → {name_of(psi)}"); _axis(ax, n)
        path = np.array([rotate(v0, n, a) for a in np.linspace(0, th, 60)])
        ax.plot(path[:, 0], path[:, 1], path[:, 2], color=ORANGE, lw=2.2, ls=":")
        _arrow(ax, v0, GRAY); _arrow(ax, state_to_bloch(psi), BLUE)
    plt.show()
    return psi

final = plot_sequence(["H", "Z", "H"])
print("H→Z→H sonucu:", np.round(final, 3), " = X|0⟩ ?", same_state(final, X @ STATES["|0⟩"]))

In [ ]:
# Aynı yolculuğun Qiskit karşılığı
qc = QuantumCircuit(1); qc.h(0); qc.z(0); qc.h(0)
display(qc.draw("mpl", style=CSTYLE))
print(Statevector(qc))

---
## H · Tersinirlik: kapıyı geri almak
Her kapı üniterdir, dolayısıyla tersi U† vardır. X, Y, Z, H için **U† = U**: aynı kapıyı bir kez daha uygulamak onu geri alır. Bir kapı **dizisini** geri almak için ise kapıları **ters sırada** uygulamak gerekir (git'te commit'leri geri alır gibi: en son yapılan önce geri alınır).

In [ ]:
qc = QuantumCircuit(1); qc.h(0); qc.x(0)
undo = qc.inverse()                      # Qiskit ters devreyi otomatik üretir
full = qc.compose(undo)
display(full.draw("mpl", style=CSTYLE))
print("U† U = I ?", Operator(full).equiv(Operator(QuantumCircuit(1))))

# Yanlış sıra ile geri alma denemesi:
wrong = qc.compose(qc)                   # aynı sırayla tekrar: H X H X
print("Aynı sırayla tekrar = I ?", Operator(wrong).equiv(Operator(QuantumCircuit(1))))
print("HXHX = ZX, yani I değil! Doğru geri alma: X sonra H (ters sıra).")

---
## I · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Etki tablosu fonksiyonu
`effect_map(U)` bir sözlük döndürsün: anahtar girdi durumunun adı, değer çıktının adı (**global faz yok sayılarak**; `same_state` kullanın). Ör. X için `{"|0⟩": "|1⟩", ...}`.

In [ ]:
def effect_map(U):
    # TODO: STATES üzerinde dolaşın, U @ s sonucunu STATES içindeki durumlarla karşılaştırın
    pass

assert effect_map(X)["|0⟩"] == "|1⟩" and effect_map(X)["|+⟩"] == "|+⟩"
assert effect_map(Z) == {"|0⟩": "|0⟩", "|1⟩": "|1⟩", "|+⟩": "|−⟩", "|−⟩": "|+⟩", "|+i⟩": "|−i⟩", "|−i⟩": "|+i⟩"}
assert effect_map(H)["|+i⟩"] == "|−i⟩"
print("Alıştırma 1 ✓")

### Alıştırma 2 · Kapı dedektifi 🔍
Aşağıda üç "gizemli" matris var. Her biri X, Y, Z, H'den biridir, ama **bilinmeyen bir global fazla** çarpılmıştır. `identify(U)` fonksiyonu yalnızca **altı durum üzerindeki etkiye** (Alıştırma 1) bakarak kapının adını döndürsün.

In [ ]:
M1 = np.exp(1j*0.7) * np.array([[0, 1], [1, 0]])
M2 = 1j * np.array([[1, 1], [1, -1]]) / np.sqrt(2)
M3 = -np.array([[0, -1j], [1j, 0]])

def identify(U):
    # TODO: effect_map(U) ile her aday kapının effect_map'ini karşılaştırın
    pass

assert identify(M1) == "X" and identify(M2) == "H" and identify(M3) == "Y"
assert identify(np.exp(-1j*2.1) * Z) == "Z"
print("Alıştırma 2 ✓")

### Alıştırma 3 · Dönüş eksenini bul
Bir kapının dönüş ekseni, **kapının yerinde bıraktığı Bloch noktasıdır**. Yerinde kalan durum, U·v = λ·v eşitliğini sağlayan v vektörüdür (λ yalnızca global faz); buna **özvektör** denir. `fixed_axis(U)` fonksiyonu `np.linalg.eig` ile bir özvektör bulup onun Bloch koordinatını döndürsün. İşaret önemli değil (n ve −n aynı eksendir).

In [ ]:
def fixed_axis(U):
    # TODO: özvektörlerden birini alın, state_to_bloch ile Bloch koordinatına çevirin
    pass

def same_axis(a, b): return np.allclose(a, b, atol=1e-6) or np.allclose(a, -np.asarray(b), atol=1e-6)
assert same_axis(fixed_axis(X), [1, 0, 0]) and same_axis(fixed_axis(Y), [0, 1, 0]) and same_axis(fixed_axis(Z), [0, 0, 1])
assert same_axis(fixed_axis(H), [1/np.sqrt(2), 0, 1/np.sqrt(2)])
print("Alıştırma 3 ✓")

### Alıştırma 4 · Dönüş yolunu tahmin et (kalem-kağıt, sonra kod)
Önce **kodu çalıştırmadan** aşağıdaki dizilerin |0⟩'dan başlayınca varacağı Bloch noktasını tahmin edin ve `tahmin` sözlüğüne yazın (x, y, z tam sayılar). Sonra `assert` ile kontrol edin; `plot_sequence` ile yolculuğu izleyin.

In [ ]:
diziler = {"a": ["H", "Z"], "b": ["X", "H"], "c": ["H", "Y"], "d": ["Y", "H", "Z"]}
tahmin = {
    "a": (0, 0, 0),   # TODO
    "b": (0, 0, 0),   # TODO
    "c": (0, 0, 0),   # TODO
    "d": (0, 0, 0),   # TODO
}
for k, gs in diziler.items():
    psi = STATES["|0⟩"]
    for g in gs: psi = GATES[g] @ psi
    assert np.allclose(state_to_bloch(psi), tahmin[k]), f"{k} dizisi yanlış: {np.round(state_to_bloch(psi), 3)}"
print("Alıştırma 4 ✓")

### Alıştırma 5 · Film şeridi kareleri
`rotation_frames(v, n, degs)` bir Bloch vektörünün n ekseni etrafında verilen açılardaki (derece) konumlarını liste olarak döndürsün (`rotate` kullanın). |0⟩'ı x ekseni etrafında 90° döndürünce hangi durum elde edilir?

In [ ]:
def rotation_frames(v, n, degs):
    # TODO
    pass

fr = rotation_frames([0, 0, 1], [1, 0, 0], [0, 90, 180])
assert np.allclose(fr[0], [0, 0, 1]) and np.allclose(fr[1], [0, -1, 0]) and np.allclose(fr[2], [0, 0, -1])
fr_h = rotation_frames([0, 0, 1], [1/np.sqrt(2), 0, 1/np.sqrt(2)], [180])
assert np.allclose(fr_h[0], [1, 0, 0])
print("Alıştırma 5 ✓  — x etrafında 90°: |0⟩ → |−i⟩ (yarım X)")

### Alıştırma 6 · Özdeşlik doğrulayıcı
`check(lhs, rhs)` iki kapı dizisini (ör. `"hxh"`, `"z"`; **devre sırası**, soldan sağa) Qiskit `Operator` ile karşılaştırsın ve `(equiv, exact)` döndürsün: `equiv` global faz hariç, `exact` birebir eşitlik.

In [ ]:
def check(lhs, rhs):
    # TODO: circ(...) ve Operator(...).equiv / == kullanın
    pass

assert check("hxh", "z") == (True, True)
assert check("hyh", "y") == (True, False)      # HYH = −Y
assert check("xz", "zx") == (True, False)      # XZ = −ZX
assert check("hz", "zh") == (False, False)     # HZ ≠ ZH
print("Alıştırma 6 ✓")

### Alıştırma 7 · Geri alma (undo)
`undo_ops(ops)` bir kapı dizisini (ör. `["h", "z", "x"]`) geri alan diziyi döndürsün. X, Y, Z, H kendi tersleri olduğu için yalnızca **sırayı ters çevirmek** yeter. Sonucu Qiskit'in `qc.inverse()` çıktısıyla karşılaştırın.

In [ ]:
def undo_ops(ops):
    # TODO
    pass

ops = ["h", "z", "x", "y", "h"]
u = undo_ops(ops)
assert Operator(circ(ops + u)).equiv(Operator(QuantumCircuit(1)))
assert Operator(circ(u)) == Operator(circ(ops).inverse())
print("Alıştırma 7 ✓", u)

### Alıştırma 8 · En kısa yol bulmacası
`shortest(start, target)` yalnızca {X, Y, Z, H} kapılarını kullanarak `start` durumunu `target` durumuna (global faz hariç) götüren **en kısa kapı dizisini** genişlik öncelikli arama (BFS) ile bulsun. Birden fazla en kısa dizi olabilir; uzunluk önemlidir.

In [ ]:
from collections import deque

def shortest(start, target, max_len=4):
    # TODO: kuyruğa (durum, dizi) çiftleri koyun; her adımda 4 kapıyı deneyin
    pass

assert len(shortest("|0⟩", "|1⟩")) == 1
assert len(shortest("|0⟩", "|−⟩")) == 2
assert len(shortest("|+i⟩", "|−i⟩")) == 1
assert shortest("|+⟩", "|+⟩") == []
print("Alıştırma 8 ✓", shortest("|0⟩", "|−⟩"))

---
### Haftanın özeti
- Her tek kübit kapısı = üniter 2×2 matris = **Bloch küresinde bir eksen etrafında dönüş**
- X: x ekseni 180° (NOT) · Y: y ekseni 180° · Z: z ekseni 180° (işaret çevirme) · H: (x+z)/√2 ekseni 180° (baz değiştirici)
- Eksen üzerindeki durumlar yerinde kalır (yalnızca global faz alır)
- X² = Y² = Z² = H² = I; HXH = Z; HZH = X; XZ = −ZX; XY = iZ; HYH = −Y
- `Operator(a).equiv(Operator(b))` global fazı yok sayar, `==` saymaz
- Bir diziyi geri almak için kapıları **ters sırada** uygula (`qc.inverse()`)
- X, Y, Z, H ile yalnızca altı temel durum arasında zıplanabilir; |0⟩ → |+i⟩ bile ulaşılamaz

**Gelecek hafta:** Tek kübit kapıları II — S, T, Rx, Ry, Rz ve U kapıları; **istediğiniz açıyla** dönüş, kapıları birleştirme, devre derinliği ve `transpile`; "hedef duruma ulaş" bulmacaları.